In [4]:
#Bloco 1 — Importar bibliotecas e carregar a feature table
import pandas as pd
import numpy as np

# Carrega a feature table em CSV
features_df = pd.read_csv('../outputs/feature_table.csv')

print(f"Shape: {features_df.shape}")
print(features_df.head())


Shape: (1388, 13)
     Item  avg_monthly  std_monthly      CV      CV2   ADI  zero_ratio  \
0  300007        50.71       243.19  4.7958  23.0000  24.0      0.9583   
1  300019       574.46      1793.44  3.1220   9.7466   8.0      0.8750   
2  300021         8.79        42.16  4.7958  23.0000  24.0      0.9583   
3  300027        28.58       137.08  4.7958  23.0000  24.0      0.9583   
4  300042       169.92       621.64  3.6585  13.3847   8.0      0.8750   

   D_annual  avg_order_size  trend_slope  n_clientes  n_pedidos  Classe  
0     608.5         1217.00       6.8787           1          1      30  
1    6893.5         4595.67    -115.0743           6          7      30  
2     105.5          211.00      -2.1100           1          1      30  
3     343.0          686.00       5.6670           1          1      30  
4    2039.0         1359.33     -27.6165           2          3      30  


In [5]:
# Classificação pelo Eixo 1 — ADI e CV²
def classify_demand(adi, cv2):
    if adi < 1.32 and cv2 < 0.49:
        return 'stable'
    elif adi < 1.32 and cv2 >= 0.49:
        return 'erratic'
    elif adi >= 1.32 and cv2 < 0.49:
        return 'intermittent'
    else:
        return 'lumpy'

features_df['demand_class'] = features_df.apply(
    lambda row: classify_demand(row['ADI'], row['CV2']), axis=1
)

print("Distribuição por tipo de demanda:")
print(features_df['demand_class'].value_counts())
print(f"\nTotal: {len(features_df)} itens")

Distribuição por tipo de demanda:
demand_class
lumpy      1142
stable      139
erratic     107
Name: count, dtype: int64

Total: 1388 itens


In [6]:
#Bloco 3 — Classificação pelo Eixo 2 — Volume e número de clientes
# Calcula os thresholds pela mediana
threshold_volume   = features_df['D_annual'].median()
threshold_clientes = features_df['n_clientes'].median()

print(f"Threshold volume anual: {threshold_volume:.1f} kg")
print(f"Threshold nº clientes: {threshold_clientes:.1f}")

# Classifica cada item
def classify_commercial(d_annual, n_clientes, thresh_vol, thresh_cli):
    volume = 'high' if d_annual >= thresh_vol else 'low'
    clients = 'many' if n_clientes >= thresh_cli else 'few'
    return f"{volume}_volume_{clients}_clients"

features_df['commercial_class'] = features_df.apply(
    lambda row: classify_commercial(
        row['D_annual'], row['n_clientes'],
        threshold_volume, threshold_clientes
    ), axis=1
)

print("\nDistribuição por perfil comercial:")
print(features_df['commercial_class'].value_counts())

Threshold volume anual: 4513.5 kg
Threshold nº clientes: 2.0

Distribuição por perfil comercial:
commercial_class
high_volume_many_clients    590
low_volume_few_clients      422
low_volume_many_clients     272
high_volume_few_clients     104
Name: count, dtype: int64


In [7]:
#Bloco 4 — Combinar os dois eixos para gerar os 5 clusters finais
def assign_cluster(demand_class, commercial_class):
    
    # MTO puro — lumpy com baixo volume e poucos clientes
    if demand_class == 'lumpy' and commercial_class == 'low_volume_few_clients':
        return 'MTO'
    
    # MTS — estável ou errático com alto volume e muitos clientes
    if demand_class in ['stable', 'erratic'] and commercial_class == 'high_volume_many_clients':
        return 'MTS'
    
    # Comakership — estável com alto volume e poucos clientes
    if demand_class == 'stable' and commercial_class == 'high_volume_few_clients':
        return 'Comakership'
    
    # Batch-to-Order — intermitente ou baixo volume com muitos clientes
    if demand_class == 'intermittent':
        return 'Batch-to-Order'
    
    if demand_class in ['stable', 'erratic'] and commercial_class == 'low_volume_many_clients':
        return 'Batch-to-Order'
    
    # Dynamic MTS/MTO — todos os restantes
    return 'Dynamic'

features_df['policy_cluster'] = features_df.apply(
    lambda row: assign_cluster(row['demand_class'], row['commercial_class']), axis=1
)

print("Distribuição por cluster de política:")
print(features_df['policy_cluster'].value_counts())
print(f"\nTotal: {len(features_df)} itens")

Distribuição por cluster de política:
policy_cluster
Dynamic        721
MTO            422
MTS            241
Comakership      4
Name: count, dtype: int64

Total: 1388 itens


In [9]:
#Bloco 5 — Verificar alguns itens específicos
# Verifica itens conhecidos
itens_check = ['300091', '300007', '300092', '300019']

check = features_df[features_df['Item'].astype(str).isin(itens_check)][
    ['Item', 'ADI', 'CV2', 'D_annual', 'n_clientes', 
     'demand_class', 'commercial_class', 'policy_cluster']
]

print(check.to_string(index=False))

  Item     ADI     CV2  D_annual  n_clientes demand_class         commercial_class policy_cluster
300007 24.0000 23.0000     608.5           1        lumpy   low_volume_few_clients            MTO
300019  8.0000  9.7466    6893.5           6        lumpy high_volume_many_clients        Dynamic
300091  1.2632  0.7066    9398.0          26      erratic high_volume_many_clients            MTS
300092  3.4286  5.7513    3102.5           7        lumpy  low_volume_many_clients        Dynamic


In [10]:
#Bloco 6 — Salvar o resultado
features_df.to_csv('../outputs/clustering_table.csv', index=False)
print("Clustering table salva com sucesso.")
print(f"\nResumo final:")
print(features_df['policy_cluster'].value_counts())

Clustering table salva com sucesso.

Resumo final:
policy_cluster
Dynamic        721
MTO            422
MTS            241
Comakership      4
Name: count, dtype: int64
